In [1]:
import os 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import f1_score, classification_report
import pandas as pd
from PIL import Image

# EfficientNet Pre-trained Model

### Konfigurasi parameter

In [3]:
MODEL_NAME = "efficientnet"
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 1e-4

TRAIN_DIR = "dataset_split/train"
VAL_DIR = "dataset_split/val"
BEST_MODEL_PATH = f"best_{MODEL_NAME}.pth"


### Membangun arsitektur model dan memodifikasi layer klasifikasi akhir

In [14]:
def get_model(model_name, num_classes=3):
    if model_name == "efficientnet":
        model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "convnext":
        model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    return model   

### Proses utama training model EfficientNet

In [10]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Memulai proses training di device: {device}")
    
    # Data Augmentation (Train) & Normalisasi (Val)
    train_transforms = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
    val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transforms)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = get_model(MODEL_NAME).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scaler = torch.amp.GradScaler(device='cuda') # AMP untuk efisiensi VRAM RTX 4060
    
    best_f1 = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * inputs.size(0)
            
        # Validasi
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                
        epoch_f1 = f1_score(val_labels, val_preds, average='macro')
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_dataset):.4f} | Val F1: {epoch_f1:.4f}")
        
        if epoch_f1 > best_f1:
            best_f1 = epoch_f1
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"Model {BEST_MODEL_PATH} tersimpan")
            
    # Evaluasi Akhir (Classification Report)
    print("\nMembuat classification report dari model terbaik\n")
    
    # 1. Muat kembali bobot model terbaik yang baru saja disimpan
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    model.eval()
    
    final_preds = []
    final_labels = []
    
    # 2. Lakukan prediksi ulang pada seluruh data validasi
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            final_preds.extend(preds.cpu().numpy())
            final_labels.extend(labels.cpu().numpy())
            
    # 3. Ambil nama kelas langsung dari folder dataset (misal: Electronic, Organic, Recycable)
    class_names = val_dataset.classes
    
    # 4. Cetak laporan lengkap
    report = classification_report(final_labels, final_preds, target_names=class_names, digits=4)
    print(report)
            
if __name__ == "__main__":
    main()

Memulai proses training di device: cuda


C:\Users\keaga\AppData\Local\Temp\ipykernel_14576\3766845227.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/30 | Loss: 0.2596 | Val F1: 0.9428
Model best_efficientnet.pth tersimpan


C:\Users\keaga\AppData\Local\Temp\ipykernel_14576\3766845227.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 2/30 | Loss: 0.1482 | Val F1: 0.9323


...

Epoch 28/30 | Loss: 0.0298 | Val F1: 0.9294


C:\Users\keaga\AppData\Local\Temp\ipykernel_14576\3766845227.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 29/30 | Loss: 0.0284 | Val F1: 0.9396


C:\Users\keaga\AppData\Local\Temp\ipykernel_14576\3766845227.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 30/30 | Loss: 0.0266 | Val F1: 0.9359

Membuat classification report dari model terbaik

              precision    recall  f1-score   support

0_Recyclable     0.9376    0.9305    0.9341      1987
1_Electronic     0.9552    0.9689    0.9620       771
   2_Organic     0.9496    0.9511    0.9503      2494

    accuracy                         0.9459      5252
   macro avg     0.9475    0.9502    0.9488      5252
weighted avg     0.9459    0.9459    0.9459      5252



# ConvNext Pre-trained Model

### Konfigurasi parameter

In [11]:
MODEL_NAME = "convnext"
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 1e-4

TRAIN_DIR = "dataset_split/train"
VAL_DIR = "dataset_split/val"
BEST_MODEL_PATH = f"best_{MODEL_NAME}.pth"


### Proses utama training model ConvNext

In [15]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Memulai proses training di device: {device}")
    
    # Data Augmentation (Train) & Normalisasi (Val)
    train_transforms = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
    val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transforms)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = get_model(MODEL_NAME).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scaler = torch.amp.GradScaler(device='cuda') # AMP untuk efisiensi VRAM RTX 4060
    
    best_f1 = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * inputs.size(0)
            
        # Validasi
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                
        epoch_f1 = f1_score(val_labels, val_preds, average='macro')
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_dataset):.4f} | Val F1: {epoch_f1:.4f}")
        
        if epoch_f1 > best_f1:
            best_f1 = epoch_f1
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"Model {BEST_MODEL_PATH} tersimpan")
            
    # Evaluasi Akhir (Classification Report)
    print("\nMembuat classification report dari model terbaik\n")
    
    # 1. Muat kembali bobot model terbaik yang baru saja disimpan
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    model.eval()
    
    final_preds = []
    final_labels = []
    
    # 2. Lakukan prediksi ulang pada seluruh data validasi
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            final_preds.extend(preds.cpu().numpy())
            final_labels.extend(labels.cpu().numpy())
            
    # 3. Ambil nama kelas langsung dari folder dataset (misal: Electronic, Organic, Recycable)
    class_names = val_dataset.classes
    
    # 4. Cetak laporan lengkap
    report = classification_report(final_labels, final_preds, target_names=class_names, digits=4)
    print(report)
            
if __name__ == "__main__":
    main()

Memulai proses training di device: cuda


C:\Users\keaga\AppData\Local\Temp\ipykernel_14576\3766845227.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/30 | Loss: 0.2137 | Val F1: 0.9293
Model best_convnext.pth tersimpan


C:\Users\keaga\AppData\Local\Temp\ipykernel_14576\3766845227.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 2/30 | Loss: 0.1247 | Val F1: 0.9422
Model best_convnext.pth tersimpan


...

Epoch 28/30 | Loss: 0.0259 | Val F1: 0.9363


C:\Users\keaga\AppData\Local\Temp\ipykernel_14576\3766845227.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 29/30 | Loss: 0.0259 | Val F1: 0.9439


C:\Users\keaga\AppData\Local\Temp\ipykernel_14576\3766845227.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 30/30 | Loss: 0.0222 | Val F1: 0.9461

Membuat classification report dari model terbaik

              precision    recall  f1-score   support

0_Recyclable     0.9492    0.9396    0.9444      1987
1_Electronic     0.9460    0.9767    0.9611       771
   2_Organic     0.9614    0.9595    0.9605      2494

    accuracy                         0.9545      5252
   macro avg     0.9522    0.9586    0.9553      5252
weighted avg     0.9545    0.9545    0.9545      5252



# Inference & Submission

In [ ]:
# Konfigurasi Prediksi
MODEL_NAME = "efficientnet" # pastikan sama dengan model yang ditraining
BATCH_SIZE = 16
TEST_DIR = "data/test"
SUBMISSION_TEMPLATE = "submission.csv"
OUTPUT_SUBMISSION = f"submission_final_{MODEL_NAME}.csv"
SAVE_MODEL_PATH = f"best_{MODEL_NAME}.pth"

In [ ]:
class TestDataset(Dataset):
    """Dataset kustom untuk membaca folder tanpa label kategori"""
    def __init__(self, img_ids, test_dir, transform=None):
        self.img_ids = img_ids,
        self.test_dir = test_dir,
        self.transform = transform
        
    def __len__(self):
        return len(self.img_ids)
    
    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_path = os.path.join(self.test_dir, f"{img_id}.jpg")
        image = Image.open(img_path.convert("RGB"))
        if self.transform:
            image = self.transform(image)
        return image, img_id

In [ ]:
def main_submission():
    device = torch.devie("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Memulai proses inferensi menggunakan device: {device}")
    
    if not os.path.exists(SAVED_MODEL_PATH):
        print(f"Error: File '{SAVED_MODEL_PATH}' tidak ditemukan. Latih model terlebih dahulu")
        return
    
    # Normalisasi standar ImageNet (wajib sama dengan saat training)
    test_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Membangun model dan membuat beban (weights) hasil training
    model = get_model(MODEL_NAME).to(device)
    model.load_state_dict(torch.load(SAVED_MODEL_PATH))
    model.eval()
    
    # Memuat template submission.csv
    sub_df = pd.read_csv(SUBMISSION_TEMPLATE)
    
    test_dataset = TestDataset(img_ids=sub_df['id'].values, test_dir=TEST_DIR, transform=test_transforms)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    predictions = []
    
    print("Memproses gambar untuk submission...")
    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            
            # Megambil index class dengan probabilitas tertinggi
            _, preds = torch.max(outputs, 1)
            predictions.extend(preds.cpu().numpy())
            
    # Menyimpan hasil ke file CSV baru
    sub_df['predicted'] = predictions
    sub_df.to_csv(OUTPUT_SUBMISSION, index=False)
    print(f"Selesai! Prediksi berhasil disimpan di '{OUTPUT_SUBMISSION}'")

if __name__ == "__main__":
    main_submission()